In [2]:
# Librerías para manipulación, visualización y carga portable de datos.
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import os

# Usamos una ruta para Kaggle y otra para ejecución local en VS Code.
kaggle_path = '/kaggle/input/datasets/olistbr/brazilian-ecommerce'
local_path = 'data'

# Cargamos el dataset desde la ruta que exista en el entorno actual.
if os.path.exists(kaggle_path):
    base_path = kaggle_path
    print("Cargado desde el entorno de Kaggle")
elif os.path.exists(local_path):
    base_path = local_path
    print("Cargado desde el entorno local (VS Code)")
else:
    raise FileNotFoundError("No se encontro el archivo en ninguna de las rutas.")

# Funcion para carga dinamica
load_olist_dataset = lambda name: pd.read_csv(f"{base_path}/olist_{name}_dataset.csv")

# Carga de datasets
orders      = load_olist_dataset("orders")
items       = load_olist_dataset("order_items")
payments    = load_olist_dataset("order_payments")
reviews     = load_olist_dataset("order_reviews")
customers   = load_olist_dataset("customers")
products    = load_olist_dataset("products")
sellers     = load_olist_dataset("sellers")
geo         = load_olist_dataset("geolocation")
cat_tr      = pd.read_csv(f"{base_path}/product_category_name_translation.csv")

# Conversion de fechas a datetime
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

# Sanity check
for name, df in {"orders": orders, "items": items, "customers": customers}.items():
    print(name, df.shape, "| nulos:", df.isna().sum().sum())

Cargado desde el entorno de Kaggle
orders (99441, 8) | nulos: 4908
items (112650, 7) | nulos: 0
customers (99441, 5) | nulos: 0


In [3]:
cust_orders = orders.merge(customers, on="customer_id")
freq = cust_orders.groupby("customer_unique_id")["order_id"].nunique()
print(freq.value_counts(normalize=True).head())

order_id
1    0.968812
2    0.028565
3    0.002112
4    0.000312
5    0.000083
Name: proportion, dtype: float64


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def build_preprocessor(num_cols, cat_cols):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frecuency")),
        ("ohe",     OneHotEncoder(handle_unknown="ignore")),                
    ])
    
    return ColumnTransformer([
        ("nume", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])